# 01 — Extract Frames → YOLO Dataset
Put videos in `data/raw/videos/` before running.

In [ ]:
# !pip install opencv-python tqdm

In [ ]:
import os, cv2, random, shutil, pathlib
from tqdm import tqdm

RAW_VIDEOS_DIR = 'data/raw/videos'
OUT_ROOT = 'data/processed/visdrone_like'
FPS_TARGET = 5.0
TRAIN_SPLIT = 0.9

os.makedirs(RAW_VIDEOS_DIR, exist_ok=True)

def ensure_dirs(root):
    for sub in ['images/train','images/val','labels/train','labels/val']:
        os.makedirs(os.path.join(root, sub), exist_ok=True)

def extract_video(video_path, out_dir, fps_target):
    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f'Cannot open {video_path}'
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(int(round(fps / fps_target)), 1)
    idx = 0; saved = []
    pbar = tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), desc=os.path.basename(video_path), unit='f')
    while True:
        ok, frame = cap.read()
        if not ok: break
        if idx % step == 0:
            name = f"{pathlib.Path(video_path).stem}_{idx:06d}.jpg"
            fp = os.path.join(out_dir, name)
            cv2.imwrite(fp, frame)
            saved.append(fp)
        idx += 1
        pbar.update(1)
    pbar.close()
    cap.release()
    return saved

ensure_dirs(OUT_ROOT)
tmp = os.path.join(OUT_ROOT, 'images', 'all')
os.makedirs(tmp, exist_ok=True)

for f in os.listdir(RAW_VIDEOS_DIR):
    if f.lower().endswith(('.mp4','.mov','.avi','.mkv')):
        saved = extract_video(os.path.join(RAW_VIDEOS_DIR, f), tmp, FPS_TARGET)
        print(f'Extracted {len(saved)} frames from {f}')

imgs = [os.path.join(tmp, x) for x in os.listdir(tmp) if x.lower().endswith('.jpg')]
random.shuffle(imgs)
n_train = int(len(imgs) * TRAIN_SPLIT)
for i, fp in enumerate(imgs):
    split = 'train' if i < n_train else 'val'
    out_img = os.path.join(OUT_ROOT, 'images', split, os.path.basename(fp))
    shutil.move(fp, out_img)
    lbl = os.path.join(OUT_ROOT, 'labels', split, os.path.splitext(os.path.basename(fp))[0] + '.txt')
    open(lbl, 'a').close()

shutil.rmtree(tmp, ignore_errors=True)
print('Dataset ready at', OUT_ROOT)